In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
%matplotlib inline
import streamlit as st
import mysql.connector
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String

pd.options.display.max_rows = 9999

In [2]:
pisa2022_school_df = pd.read_csv(r"C:\1stApril\Work\4rd_Year\4rd2\CSS400_Project_Development\CN1-2023\pisa2022_analysis\StreamlitDev\pisa2022.csv", encoding='cp1252')
pisa2022_school_df

,ï»¿CNT,CNTRYID,CNTSCHID,CNTSTUID,CYC,NatCen,STRATUM,SUBNATIO,REGION,OECD,...,PV6MPRE,PV7MPRE,PV8MPRE,PV9MPRE,PV10MPRE,SENWT,i,Country_Region,Ranking,AverageScore
0,ALB,8,800282,800001,08MS,800,ALB03,80000,800,NON-OECD,...,331.017,223.752,305.671,230.156,289.436,0.55561,218,Albania,70,367.3
1,ALB,8,800115,800002,08MS,800,ALB03,80000,800,NON-OECD,...,284.836,364.565,304.044,347.626,352.269,0.76431,218,Albania,70,367.3
2,ALB,8,800242,800003,08MS,800,ALB01,80000,800,NON-OECD,...,289.896,338.469,316.296,324.361,343.351,1.37877,218,Albania,70,367.3
3,ALB,8,800245,800005,08MS,800,ALB08,80000,800,NON-OECD,...,165.575,246.156,238.322,275.860,227.466,1.49361,218,Albania,70,367.3
4,ALB,8,800285,800006,08MS,800,ALB03,80000,800,NON-OECD,...,461.793,514.465,510.462,490.537,503.793,0.65249,218,Albania,70,367.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613739,UZB,860,86000120,86007488,08MS,86000,UZB26,8600000,86000,NON-OECD,...,386.969,376.271,368.287,395.329,278.263,0.71987,218,Uzbekistan,78,351.7
613740,UZB,860,86000140,86007489,08MS,86000,UZB04,8600000,86000,NON-OECD,...,179.328,233.470,203.001,254.340,233.187,0.65541,218,Uzbekistan,78,351.7
613741,UZB,860,86000024,86007490,08MS,86000,UZB10,8600000,86000,NON-OECD,...,387.329,456.236,424.790,408.986,460.542,0.67910,218,Uzbekistan,78,351.7
613742,UZB,860,86000174,86007491,08MS,86000,UZB16,8600000,86000,NON-OECD,...,248.165,405.860,364.173,346.663,341.668,0.68618,218,Uzbekistan,78,351.7


In [3]:
pisa2022_school_df.dtypes

ï»¿CNT             object
CNTRYID             int64
CNTSCHID            int64
CNTSTUID            int64
CYC                object
NatCen              int64
STRATUM            object
SUBNATIO            int64
REGION              int64
OECD               object
ADMINMODE           int64
LANGTEST_QQQ        int64
LANGTEST_COG        int64
Option_ICTQ         int64
Option_WBQ          int64
Option_PQ           int64
Option_TQ           int64
Option_UH           int64
BOOKID              int64
ST001D01T           int64
ST003D02T           int64
ST003D03T           int64
ST004D01T           int64
ST250Q01JA          int64
ST250Q02JA          int64
ST250Q03JA          int64
ST250Q04JA          int64
ST250Q05JA          int64
ST250D06JA          int64
ST250D07JA          int64
ST251Q01JA          int64
ST251Q02JA          int64
ST251Q03JA          int64
ST251Q04JA          int64
ST251Q06JA          int64
ST251Q07JA          int64
ST251D08JA          int64
ST251D09JA          int64
ST253Q01JA  

In [4]:
replacements = {
    'object' : 'varchar(20)',
    'float64' : 'float',
    'init64' : 'int',
    'datetime64': 'timestamp'
}

col_str = ", ".join("{} {}".format(n,d) for (n,d) in zip(pisa2022_school_df.columns, pisa2022_school_df.dtypes.replace(replacements)))
col_str

'ï»¿CNT varchar(20), CNTRYID int64, CNTSCHID int64, CNTSTUID int64, CYC varchar(20), NatCen int64, STRATUM varchar(20), SUBNATIO int64, REGION int64, OECD varchar(20), ADMINMODE int64, LANGTEST_QQQ int64, LANGTEST_COG int64, Option_ICTQ int64, Option_WBQ int64, Option_PQ int64, Option_TQ int64, Option_UH int64, BOOKID int64, ST001D01T int64, ST003D02T int64, ST003D03T int64, ST004D01T int64, ST250Q01JA int64, ST250Q02JA int64, ST250Q03JA int64, ST250Q04JA int64, ST250Q05JA int64, ST250D06JA int64, ST250D07JA int64, ST251Q01JA int64, ST251Q02JA int64, ST251Q03JA int64, ST251Q04JA int64, ST251Q06JA int64, ST251Q07JA int64, ST251D08JA int64, ST251D09JA int64, ST253Q01JA int64, ST254Q01JA int64, ST254Q02JA int64, ST254Q03JA int64, ST254Q04JA int64, ST254Q05JA int64, ST254Q06JA int64, ST255Q01JA int64, ST256Q01JA int64, ST256Q02JA int64, ST256Q03JA int64, ST256Q06JA int64, ST256Q07JA int64, ST256Q08JA int64, ST256Q09JA int64, ST256Q10JA int64, ST230Q01JA int64, ST005Q01JA int64, ST006Q01JA 

In [7]:
username = 'root'
password = 'Owen01042545'
hostname = 'localhost'  # Usually 'localhost' if MySQL is running locally
database_name = 'pisa2022'


# Create connection string
connection_str = f'mysql+mysqlconnector://{username}:{password}@{hostname}/{database_name}'

# Connect to MySQL
engine = create_engine(connection_str)
connection = engine.connect()

# Example SQL query
query = 'SELECT * FROM pisa2022_data'
df = pd.read_sql(query, connection)

# Close connection
connection.close()

# Display the dataframe
df.head()

,CNT,CNTRYID,CNTSCHID,CNTSTUID,CYC,NatCen,STRATUM,SUBNATIO,REGION,OECD,...,PV6MPRE,PV7MPRE,PV8MPRE,PV9MPRE,PV10MPRE,SENWT,i,CountryRegion,Rank_col,Average_Score


In [9]:
pisa2022_school_df.to_sql('pisa2022_data', con=engine, if_exists='append', index=False)